# **基础作业：不同参数规模的Embedding模型对RAG系统性能影响的探究**  

## **实验介绍**  

本项目旨在探究在检索增强生成（RAG）系统中，Embedding模型（文本嵌入模型）的参数规模对系统整体性能的影响。文本嵌入的质量直接决定了知识库检索的精准度，进而影响大语言模型生成答案的相关性和准确性。  

本次实验基于一个电商智能问答助手RAG系统，该系统原始版本（基线）采用的是轻量级的 `Qwen/Qwen3-Embedding-0.6B` 模型。为了评估模型规模带来的变化，本次实验将Embedding模型替换为参数量更大的 `Qwen/Qwen3-Embedding-4B` 模型，同时保持RAG系统的其他组件（如文档分块策略、TF-IDF与向量混合检索的框架、以及生成答案所用的`Qwen3-4B`大语言模型）完全不变。  

为保证对比的公平性，**本次实验中所使用的测试问题（“退货”、“商品A的价格与性能”、“钱不够要怎么办”）与教案中“RAG实战指南”及“无RAG对比”部分所使用的示例问题完全一致。** 通过对不同版本的RAG系统以及无RAG系统进行横向比较，我们旨在全面评估更大参数量的Embedding模型在检索质量、生成答案质量上的具体表现，并论证RAG框架本身的核心价值。  


## 环境安装、配置与模型下载

In [1]:
# !pip install sentence_transformers -i https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
# !pip install --upgrade transformers==4.51.0 -i https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
# !pip install peft -i https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
# !pip install torch -i https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
# !pip install modelscope
#!modelscope download --model Qwen/Qwen3-Embedding-0.6B
# !modelscope download --model Qwen/Qwen3-4B

In [2]:
# !pip install nltk -i https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple

In [3]:
import os
import logging
# 设置 HF_ENDPOINT 环境变量
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
logging.basicConfig(level=logging.INFO)
print(os.environ.get('HF_ENDPOINT'))

https://hf-mirror.com


In [4]:
import os
import sys
import torch
import random
import numpy as np
import logging
from pathlib import Path
from collections import Counter
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
import argparse
import time # Added for performance monitoring
import re  # For text processing

# Try importing NLTK components with error handling
try:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction  # For BLEU score calculation
    NLTK_AVAILABLE = True
except ImportError:
    logging.warning("NLTK库导入失败，BLEU分数评估将被禁用")
    NLTK_AVAILABLE = False

# ========== 身份声明自动应答机制 ==========
IDENTITY_ANSWER = "您好，我是客服小助手，你问的是：\""
IDENTITY_QUESTIONS = [
    "你是什么模型", "你是谁", "你是谁的问题", "你是什么模型相关的问题", "你是什么模型相关的问题", "你是谁的问题", "你是谁", "你是什么模型"
]

def check_identity_question(query):
    for q in IDENTITY_QUESTIONS:
        if q in query:
            return True
    return False

# ========== 日志设置 ==========
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')



/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:datasets:PyTorch version 2.5.1 available.


## 数据集介绍  
使用强大的LLM来生成丰富的电商知识文档  

现在包含以下几个主要类别的内容：  
商品信息 - 添加了7种不同类型的电子产品，包含详细规格和价格  
促销政策 - 包括满减、限时特惠、会员折扣等多种促销方式  
售后服务 - 详细的退换货政策、保修条款和服务网点信息  
物流配送 - 配送范围、运费规则和特殊配送服务  
支付方式 - 多种支付选项和分期付款政策  
会员体系 - 会员等级、权益和积分规则  
常见问题 - 订单修改、账户安全等客户关注的问题  
这些内容更贴近真实电商场景，可以更好地测试和展示RAG系统的能力。

In [5]:
# ========== 数据收集与预处理 ==========
# 电商知识文档
RAW_DOCS = [
    # 商品信息
    "商品A：高性能笔记本电脑，16GB内存，512GB SSD，适合办公与游戏，售价5999元，支持分期付款。",
    "商品B：无线蓝牙耳机，降噪功能，续航30小时，适合运动与通勤，售价499元，赠送收纳盒。",
    "商品C：智能手表，支持心率监测、睡眠分析、运动追踪，防水50米，续航7天，售价1299元。",
    "商品D：家用智能扫地机器人，激光导航，APP控制，自动回充，适合各种地板清洁，售价2499元。",
    "商品E：专业级数码相机，2400万像素，4K视频拍摄，防抖功能，含18-55mm标准镜头，售价6299元。",
    "商品F：便携式蓝牙音箱，360°环绕立体声，防水防尘，续航12小时，支持TWS双音箱连接，售价299元。",
    "商品G：多功能料理机，搅拌、切碎、榨汁多合一，2000W大功率，8档调速，静音设计，售价899元。",
    
    # 促销政策
    "促销政策：满1000减100，部分商品参与，详情请咨询客服。",
    "限时特惠：每日10点、14点、20点开启秒杀，低至5折，每人限购1件。",
    "会员专享：银卡会员95折，金卡会员9折，钻石会员85折，不与其他优惠同享。",
    "新人福利：首次下单立减50元，无门槛使用，有效期7天。",
    "节日活动：618年中大促，全场商品满减，部分商品买二送一。",
    "积分兑换：消费1元积1分，积分可兑换优惠券或实物礼品，详见积分商城。",
    "优惠券规则：优惠券不可叠加使用，不可与满减活动同享，有效期请见券面说明。",
    
    # 售后服务
    "售后服务：7天无理由退换货，1年质保，支持全国联保。",
    "退货政策：商品签收后7天内可申请无理由退货，商品需保持原包装及完好，退回运费由买家承担。",
    "换货流程：联系客服提交换货申请，审核通过后寄回商品，收到退回商品后3个工作日内发出新商品。",
    "保修条款：电子产品享受1年免费保修，人为损坏、擅自拆机、进水或改装不在保修范围内。",
    "售后网点：全国设有1000+售后服务网点，可提供上门维修或到店维修服务。",
    "延长保修：可购买延长保修服务，最多可延长至3年，费用为商品价格的5%-10%。",
    
    # 物流配送
    "物流说明：订单24小时内发货，支持多家快递，包邮服务。",
    "配送范围：全国大部分地区支持配送，港澳台及偏远地区可能产生额外运费。",
    "运费规则：单笔订单满99元免运费，不满99元收取10元运费，特大件商品另计。",
    "发货时间：工作日16点前下单当天发货，节假日及特殊情况可能顺延。",
    "自提服务：支持就近门店自提，下单时选择自提点，收到提货通知后凭码取货。",
    "极速达：部分城市支持2小时极速达服务，订单满足条件可在下单页面选择。",
    
    # 支付方式
    "支付方式：支持支付宝、微信支付、银联、信用卡等多种支付方式。",
    "分期付款：单笔订单满500元可申请3-24期分期，部分银行卡用户可享免息特权。",
    "货到付款：特定区域支持货到付款服务，需支付5元手续费。",
    "发票开具：可开具电子发票或纸质发票，请在下单时选择，纸质发票将随商品一起寄出。",
    
    # 会员体系
    "会员等级：普通会员、银卡会员（累计消费5000元）、金卡会员（累计消费20000元）、钻石会员（累计消费50000元）。",
    "会员权益：专属客服、生日礼遇、提前购、专享折扣、积分加速、免费试用等，等级越高权益越多。",
    "积分规则：消费1元获得1积分，参与活动可获得额外积分，积分有效期为一年。",
    
    # 常见问题
    "订单修改：订单支付成功后，如需修改收货信息请立即联系客服，发货后无法修改。",
    "账户安全：定期修改密码，不要在不信任的设备上登录账号，警惕钓鱼网站和诈骗信息。",
    "商品咨询：关于商品参数、适用场景等问题可咨询在线客服或拨打服务热线400-888-8888。",
    "投诉建议：对服务不满意可通过APP意见反馈或发送邮件至service@example.com进行投诉。"
]

# ========== 文档分块 ==========
def chunk_docs(docs, chunk_size=50, overlap=10):
    """
    将输入的文档列表按指定的字符数进行分块（chunk），支持分块之间的重叠。

    参数说明:
        docs (List[str]): 输入的文档列表，每个元素为一个字符串，代表一篇文档。
        chunk_size (int): 每个分块的最大字符数。默认值为50。
        overlap (int): 相邻分块之间的重叠字符数。默认值为10。

    实现细节:
        - 对于每一篇文档，从头开始，按照chunk_size的长度截取一段文本作为一个分块。
        - 每次分块的起始位置向后移动(chunk_size - overlap)个字符，从而实现分块之间的重叠。
        - 如果分块的终止位置已经到达或超过文档末尾，则最后一个分块会自动截断到文档结尾。
        - 该方法适用于短文本或中等长度文档的简单分块，便于后续向量化检索。

    返回值:
        List[str]: 分块后的所有文本块组成的列表。

    示例:
        输入: ["abcdefg", "hijklmnop"], chunk_size=4, overlap=2
        输出: ['abcd', 'cdef', 'efg', 'hijk', 'ijkl', 'ijkl', 'klmn', 'mnop', 'nop']

    """
    chunks = []
    for doc in docs:
        start = 0
        while start < len(doc):
            end = min(start + chunk_size, len(doc))
            chunk = doc[start:end]
            chunks.append(chunk)
            if end == len(doc):
                break
            start += chunk_size - overlap
    return chunks

##  向量化与索引构建  

###  稀疏向量表示(TF-IDF)  
本系统使用TF-IDF算法构建稀疏向量表示：  
1. 原理：计算词频(TF)与逆文档频率(IDF)的乘积，突出重要且区分性强的词汇  
2. 实现：使用sklearn的TfidfVectorizer类，支持自动分词、停用词过滤和权重计算  
3. 优势：计算效率高，对硬件要求低，适合精确词汇匹配  
4. 局限：无法捕捉语义相似性，对同义词、上下文理解有限  

###  密集向量表示(语义嵌入)  
系统采用预训练语言模型生成密集向量表示：  
1. 模型选择：使用Qwen3-Embedding-0.6B模型，针对中文语义理解进行了优化  
2. 编码策略：对文档分块和查询使用不同的编码提示(prompt)，增强检索效果  
3. 优势：能够捕捉深层语义关系，理解同义词和上下文含义  
4. 挑战：计算资源需求较高，需要GPU加速，模型大小与推理速度需权衡  

###  混合检索策略  
本系统实现了稀疏检索与密集检索的混合策略：  
1. 权重分配：默认稀疏检索与密集检索各占50%权重，可根据应用场景调整  
2. 分数融合：线性加权组合两种检索方法的相似度分数  
3. 优势互补：稀疏检索擅长精确词匹配，密集检索擅长语义理解  
4. 适应性强：可根据不同查询类型动态调整检索策略权重  

##  向量数据库设计  

###  内存型向量存储  
当前实现使用SimpleVectorDB类作为内存型向量数据库：  
1. 数据结构：同时存储文本分块、TF-IDF矩阵和密集向量嵌入  
2. 检索方法：支持稀疏检索、密集检索和混合检索三种模式  
3. 相似度计算：使用余弦相似度衡量查询与文档的匹配程度  
4. 结果排序：按相似度分数降序排列，返回topk个最相关结果  

###  可扩展性考虑  
在实际生产环境中，可以考虑以下扩展方案：  
1. 持久化存储：使用专业向量数据库(如FAISS、Milvus、Pinecone等)替代内存存储  
2. 分布式索引：对大规模数据集采用分布式索引架构，提高检索效率  
3. 量化技术：应用向量量化(如PQ、HNSW)降低存储空间和提高检索速度  
4. 增量更新：支持知识库的增量更新，无需重建整个索引  


In [6]:
from transformers import AutoTokenizer, AutoModel
import torch
# ========== 向量化 ==========
def build_tfidf(chunks):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(chunks)
    # print("分词后的单词内容：")
    # print(vectorizer.get_feature_names_out())
    
    return vectorizer, tfidf_matrix

def build_dense_encoder():
    model_path = '/home/mw/input/Qw_Embedding_4B13681368/Qwen3-Embedding-4B'
    return SentenceTransformer(model_path)
    
def encode_dense(dense_encoder, chunks, is_query=False):
    return dense_encoder.encode(chunks, is_query=is_query)

# ========== 内存型向量数据库 ==========
class SimpleVectorDB:
    def __init__(self, chunks, tfidf_matrix, tfidf_vectorizer, dense_embeds, dense_encoder=None):
        self.chunks = chunks
        self.tfidf_matrix = tfidf_matrix
        self.tfidf_vectorizer = tfidf_vectorizer
        self.dense_embeds = dense_embeds
        self.dense_encoder = dense_encoder

    def search(self, query, topk=3, mode="hybrid", dense_encoder=None):
        results = []
        scores_sparse = None
        scores_dense = None

        # 处理 query 格式
        if isinstance(query, str):
            query_text = query
        elif isinstance(query, dict) and 'query' in query:
            query_text = query['query']
        else:
            query_text = str(query)

        # 如果未传 dense_encoder，使用对象自身的
        if dense_encoder is None and hasattr(self, 'dense_encoder'):
            dense_encoder = self.dense_encoder

        if mode in ["sparse", "hybrid"]:
            q_vec = self.tfidf_vectorizer.transform([query_text])
            scores_sparse = cosine_similarity(q_vec, self.tfidf_matrix)[0]

        # if mode in ["dense", "hybrid"] and dense_encoder is not None:
        #     # 去掉 show_progress_bar 和 prompt_name
        #     q_dense = dense_encoder.encode([query_text], is_query=True)[0]
        #     scores_dense = cosine_similarity([q_dense], self.dense_embeds)[0]

        if mode in ["dense", "hybrid"] and dense_encoder is not None:
            q_dense = dense_encoder.encode([query_text], show_progress_bar=False, prompt_name="query")[0]
            scores_dense = cosine_similarity([q_dense], self.dense_embeds)[0]


        # 融合分数
        if mode == "sparse":
            scores = scores_sparse
        elif mode == "dense":
            scores = scores_dense
        else:  # hybrid
            if scores_sparse is not None and scores_dense is not None:
                scores = 0.5 * scores_sparse + 0.5 * scores_dense
            elif scores_sparse is not None:
                scores = scores_sparse
            elif scores_dense is not None:
                scores = scores_dense
            else:
                return []

        top_idx = np.argsort(scores)[::-1][:topk]
        for idx in top_idx:
            results.append((self.chunks[idx], float(scores[idx])))
        return results



## LLM生成控制

In [7]:
# ========== 生成模块 ==========
def load_generation_model(model_path, device):
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        trust_remote_code=True,
        torch_dtype=torch.float16,
    ).to(device)
    tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        trust_remote_code=True,
    )
    return model, tokenizer

def generate_answer(model, tokenizer, context, query, device, max_new_tokens=256):
    prompt = f"""已知信息：{context}
                    用户提问：{query}
                    请基于已知信息直接回答用户问题。回答需要：
                    1. 简明扼要，不要重复已知信息
                    2. 如果已知信息中没有相关内容，明确告知用户
                    3. 不要包含任何推理过程
                    4. 不要在回答中包含"根据已知信息"之类的提示语
                    回答："""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, top_p=0.95)
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract only the answer part
    answer = full_response[len(prompt):].strip() if full_response.startswith(prompt) else full_response
    
    # Remove any remaining reasoning patterns
    answer = answer.split("回答：")[-1] if "回答：" in answer else answer
    
    return answer.strip()



##  系统评估与优化  
系统采用多维度指标评估答案质量：  
1. 事实准确率：答案中陈述的事实与知识库内容的一致性 （文本一致性）  
2. 上下文相关性：生成的答案与检索文档的相关程度（余弦相似度）

In [8]:


# ========== 系统评估与优化 ==========
def evaluate_retrieval_performance(retrieved_docs, relevant_docs):
    """
    评估检索性能
    
    参数:
        retrieved_docs (List[str]): 系统检索到的文档
        relevant_docs (List[str]): 人工标注的相关文档
    
    返回:
        Dict: 包含检索评估指标的字典
    """
    # 计算召回率
    def calculate_recall():
        relevant_retrieved = set(retrieved_docs) & set(relevant_docs)
        return len(relevant_retrieved) / len(relevant_docs) if relevant_docs else 0.0
    
    # 计算精确率
    # 精确率（Precision）是指在所有被系统检索出来的文档（retrieved_docs）中，有多少比例是真正相关的文档（relevant_docs）。
    # 计算方法是：用检索到的相关文档数量（relevant_retrieved）除以总共检索到的文档数量（retrieved_docs）。
    # 如果没有检索到任何文档，则精确率为0.0。
    def calculate_precision():
        relevant_retrieved = set(retrieved_docs) & set(relevant_docs)  # 检索结果与相关文档的交集，即被正确检索出来的相关文档
        return len(relevant_retrieved) / len(retrieved_docs) if retrieved_docs else 0.0
    
    # 计算F1分数
    def calculate_f1():
        precision = calculate_precision()
        recall = calculate_recall()
        return 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    # 计算平均排名 (MRR)
    def calculate_mrr():
        if not relevant_docs or not retrieved_docs:
            return 0.0
        
        # 找到第一个相关文档的排名
        for i, doc in enumerate(retrieved_docs):
            if doc in relevant_docs:
                return 1.0 / (i + 1)  # 排名从1开始
        return 0.0
    
    precision = calculate_precision()
    recall = calculate_recall()
    f1 = calculate_f1()
    mrr = calculate_mrr()
    
    logging.info(f"检索评估 - 精确率: {precision:.2f}, 召回率: {recall:.2f}, F1: {f1:.2f}, MRR: {mrr:.2f}")
    
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mrr": mrr
    }

def evaluate_answer_quality(answer, reference_docs, query):
    """
    评估生成答案的质量
    
    参数:
        answer (str): 生成的答案文本
        reference_docs (List[str]): 检索到的参考文档列表
        query (str): 用户的原始问题
    
    返回:
        Dict: 包含各项评估指标的字典
    """
    # 事实准确性评估（适用于中文，基于最长公共子串覆盖率）
    def check_factual_accuracy(answer, docs):
        # 合并所有参考文档为一个字符串
        doc_text = "".join(docs).replace("，", "").replace("。", "").replace("：", "").replace("！", "").replace("？", "").replace("、", "").replace("；", "")
        answer_text = answer.replace("，", "").replace("。", "").replace("：", "").replace("！", "").replace("？", "").replace("、", "").replace("；", "")

        # 如果任一为空，返回0
        if not doc_text or not answer_text:
            return 0.0

        # 以n-gram方式（如2-gram或3-gram）统计answer中有多少片段在doc_text中出现
        def get_ngrams(text, n=2):
            return {text[i:i+n] for i in range(len(text)-n+1)} if len(text) >= n else set()

        # 可以尝试2-gram和3-gram的平均
        ngram_matches = []
        ngram_total = 0
        for n in [2, 3]:
            answer_ngrams = get_ngrams(answer_text, n)
            doc_ngrams = get_ngrams(doc_text, n)
            if answer_ngrams:
                match_count = len(answer_ngrams & doc_ngrams)
                ngram_matches.append(match_count / len(answer_ngrams))
                ngram_total += 1

        # 如果没有任何ngram，返回0
        if not ngram_matches:
            return 0.0

        # 返回平均覆盖率
        return sum(ngram_matches) / ngram_total
    
    # 上下文相关性评估
    def check_context_relevance(answer, docs, query):
        # 使用向量相似度计算答案与文档的相关性
        try:
            vectorizer = TfidfVectorizer()
            docs_text = " ".join(docs)
            texts = [answer, docs_text, query]
            if all(text.strip() for text in texts):  # 确保所有文本非空
                vectors = vectorizer.fit_transform(texts)
                relevance_score = cosine_similarity(vectors[0:1], vectors[1:2])[0][0]
                return relevance_score
            else:
                print("计算相关性为0，因为所有文本为空")
        except Exception as e:
            logging.warning(f"计算相关性时出错: {str(e)}")
        return 0.0
    
    # 幻觉检测
    def check_hallucination(answer, docs):
        """检测生成内容中有多少连字出现在参考文档中的连字"""
        # 连字定义为连续两个字符的子串
        def get_bigrams(text):
            text = text.replace("，", "").replace("。", "").replace("：", "")
            return [text[i:i+2] for i in range(len(text)-1)] if len(text) >= 2 else []

        # 获取参考文档所有连字集合
        doc_text = " ".join(docs)
        doc_bigrams = set(get_bigrams(doc_text))

        # 获取答案中的连字
        answer_bigrams = get_bigrams(answer)

        if not answer_bigrams:
            return 0.0

        # 计算答案中有多少连字出现在参考文档中
        matched_bigrams = [bg for bg in answer_bigrams if bg in doc_bigrams]
        match_rate = len(matched_bigrams) / len(answer_bigrams)

        return match_rate
    
    # 计算BLEU分数（支持中文，自动按字切分）
    def calculate_bleu(answer, docs):
        """
        计算生成答案与参考文档之间的BLEU分数（支持中文，自动按字切分）。

        BLEU（Bilingual Evaluation Understudy）是一种常用的自动化评估指标，用于衡量生成文本与参考文本之间的相似度，常用于机器翻译和文本生成任务。

        具体实现步骤如下：
        1. 首先将所有参考文档（docs）合并后，按句子进行分割，得到参考句子列表 reference_sentences。
        2. 同样将生成的答案（answer）按句子分割，得到答案句子列表 answer_sentences。
        3. 对于答案中的每一个句子，进行如下操作：
            a. 将该句子分词（这里简单用 split()，假设已分好词）。
            b. 将所有参考句子也分词，作为参考（reference）列表。
            c. 使用NLTK的 sentence_bleu 函数，计算该答案句子与所有参考句子的BLEU分数。
            d. 使用 SmoothingFunction().method1 进行平滑处理，避免短句BLEU为0。
        4. 对所有答案句子的BLEU分数取平均，作为最终的BLEU分数。

        注意事项：
        - BLEU分数范围为0~1，越高表示生成内容与参考内容越接近。
        - 这里的实现是“句级BLEU”，即对每个答案句子分别计算，然后取平均。
        - 参考文档和答案都假设已经分词（如中文可用空格分词）。
        - 若NLTK不可用或输入为空，返回0.0。
        
        """
        if not NLTK_AVAILABLE:
            return 0.0

        try:
            # 1. 将参考文档分割为句子
            reference_sentences = []
            for doc in docs:
                # 按中英文标点分句
                sentences = re.split(r'[。！？!?.]', doc)
                sentences = [s.strip() for s in sentences if s.strip()]
                reference_sentences.extend(sentences)
            if not reference_sentences:
                return 0.0

            # 2. 将答案分割为句子
            answer_sentences = re.split(r'[。！？!?.]', answer)
            answer_sentences = [s.strip() for s in answer_sentences if s.strip()]
            if not answer_sentences:
                return 0.0

            # 3. 对每个答案句子计算BLEU分数（按字切分）
            bleu_scores = []
            smoothie = SmoothingFunction().method1  # 平滑处理，避免短句BLEU为0
            # 参考句子全部按字切分
            ref_tokens = [list(ref_sent) for ref_sent in reference_sentences if ref_sent]
            for ans_sent in answer_sentences:
                ans_tokens = list(ans_sent)  # 按字切分
                if ans_tokens:
                    bleu = sentence_bleu(ref_tokens, ans_tokens, smoothing_function=smoothie)
                    bleu_scores.append(bleu)
            return np.mean(bleu_scores) if bleu_scores else 0.0

        except Exception as e:
            logging.warning(f"计算BLEU分数时出错: {str(e)}")
            return 0.0
    
    # 计算困惑度 (Perplexity) - 使用简化方法评估流畅度
    def calculate_perplexity(answer):
        """使用简化方法估算困惑度 - 评估生成文本的流畅度
            在实际中，perplexity = exp(-1/N * ∑log P(w_i|context))
            其中N是词数，P(w_i|context)是每个词在上下文中的条件概率
            具体实现：
            使用预训练语言模型(如GPT、BERT)计算每个token的概率
            需要整个模型的前向推理过程
            通常基于大规模语料库训练的模型
        """
        try:
            # 简化实现：使用句子长度、标点符号比例等作为流畅度的启发式指标
            
            # 1. 检查句子长度分布 (过短或过长的句子可能不流畅)
            sentences = re.split(r'[。！？!?.]', answer)
            sentences = [s.strip() for s in sentences if s.strip()]
            if not sentences:
                return 0.0  # 没有完整句子
                
            sent_lengths = [len(s) for s in sentences]
            avg_length = np.mean(sent_lengths) if sent_lengths else 0
            
            # 长度异常惩罚：句子过短(<5)或过长(>50)会降低流畅度
            length_penalty = sum(1 for length in sent_lengths if length < 5 or length > 50) / len(sentences)
            
            # 2. 检查标点符号比例
            punctuation_marks = re.findall(r'[，。！？、：；""''（）【】《》]', answer)
            punct_ratio = len(punctuation_marks) / len(answer) if answer else 0
            
            # 标点过多或过少都会影响流畅度 (理想范围大约是 0.1-0.2)
            punct_penalty = abs(punct_ratio - 0.15) / 0.15
            
            # 3. 检查重复词语
            words = answer.split()
            word_counts = Counter(words)
            repetition_ratio = 1 - (len(word_counts) / len(words)) if words else 0
            
            # 计算综合流畅度分数 (1为最流畅，0为最不流畅)
            fluency_score = max(0, 1 - (length_penalty * 0.3 + punct_penalty * 0.3 + repetition_ratio * 0.4))
            
            # 将流畅度转换为困惑度的近似值 (困惑度越低越好)
            # 困惑度范围约定为1-50，1代表最流畅
            approx_perplexity = 1 + (1 - fluency_score) * 49
            
            return approx_perplexity
            
        except Exception as e:
            logging.warning(f"计算困惑度时出错: {str(e)}")
            return 50.0  # 返回最大困惑度值
    
    # 返回评估结果
    factual_accuracy = check_factual_accuracy(answer, reference_docs)
    print(f"factual_accuracy: {factual_accuracy}")
    context_relevance = check_context_relevance(answer, reference_docs, query)
    print(f"context_relevance: {context_relevance}")
    hallucination_rate = check_hallucination(answer, reference_docs)
    print(f"hallucination_rate: {hallucination_rate}")
    # 尝试计算BLEU分数
    bleu_score = 0.0
    try:
        bleu_score = calculate_bleu(answer, reference_docs)
    except Exception as e:
        logging.warning(f"BLEU分数计算失败: {str(e)}")
    
    # 计算困惑度
    perplexity = 50.0  # 默认值 (越高表示越不流畅)
    try:
        perplexity = calculate_perplexity(answer)
    except Exception as e:
        logging.warning(f"困惑度计算失败: {str(e)}")
    
    # 输出评估结果到日志
    logging.info(f"答案质量评估 - 事实准确性: {factual_accuracy:.2f}, 上下文相关性: {context_relevance:.2f}, " +
                 f"幻觉率: {hallucination_rate:.2f}, BLEU: {bleu_score:.4f}, 困惑度: {perplexity:.2f}")
    
    return {
        "factual_accuracy": factual_accuracy,
        "context_relevance": context_relevance,
        "hallucination_rate": hallucination_rate,
        "bleu_score": bleu_score,
        "perplexity": perplexity
    }

class PerformanceMonitor:
    """性能监控类"""
    def __init__(self):
        self.metrics = {
            "response_times": [],
            "memory_usage": [],
            "gpu_usage": [],
            "error_counts": Counter()
        }
    
    def record_response_time(self, start_time, end_time):
        """记录响应时间"""
        response_time = end_time - start_time
        self.metrics["response_times"].append(response_time)
        
    def record_resource_usage(self):
        """记录资源使用情况"""
        if torch.cuda.is_available():
            gpu_memory = torch.cuda.memory_allocated() / 1024**2  # MB
            self.metrics["gpu_usage"].append(gpu_memory)
    
    def record_error(self, error_type):
        """记录错误"""
        self.metrics["error_counts"][error_type] += 1
    
    def get_statistics(self):
        """获取统计信息"""
        if not self.metrics["response_times"]:
            return {}
            
        return {
            "avg_response_time": np.mean(self.metrics["response_times"]),
            "p95_response_time": np.percentile(self.metrics["response_times"], 95),
            "avg_gpu_usage": np.mean(self.metrics["gpu_usage"]) if self.metrics["gpu_usage"] else 0,
            "error_statistics": dict(self.metrics["error_counts"])
        }

class OptimizationManager:
    """优化管理器"""
    def __init__(self, vectordb, model, tokenizer):
        self.vectordb = vectordb
        self.model = model
        self.tokenizer = tokenizer
        self.performance_monitor = PerformanceMonitor()
        
    def optimize_retrieval_weights(self, query_set, relevant_docs):
        """优化混合检索的权重"""
        best_weights = {"sparse": 0.5, "dense": 0.5}
        best_f1 = 0
        
        for sparse_weight in np.arange(0.1, 1.0, 0.1):
            dense_weight = 1 - sparse_weight
            # self.vectordb.update_weights(sparse_weight, dense_weight) # Assuming SimpleVectorDB has an update_weights method
            
            f1_scores = []
            for query, relevant in zip(query_set, relevant_docs):
                # Pass the dense_encoder to the search method
                retrieved = self.vectordb.search(query, topk=3, dense_encoder=self.vectordb.dense_encoder)
                eval_results = evaluate_retrieval_performance(
                    [doc for doc, _ in retrieved],
                    relevant
                )
                f1_scores.append(eval_results["f1"])
            
            avg_f1 = np.mean(f1_scores)
            if avg_f1 > best_f1:
                best_f1 = avg_f1
                best_weights = {"sparse": float(sparse_weight), "dense": float(dense_weight)}
        
        return best_weights
    
    def optimize_model_inference(self):
        """优化模型推理性能"""
        # 简化模型优化，避免可能导致崩溃的量化操作
        logging.info("跳过量化，仅进行批处理优化")
        try:
            self.batch_size = self._find_optimal_batch_size()
            logging.info(f"最优批处理大小: {self.batch_size}")
        except Exception as e:
            logging.warning(f"批处理优化失败: {str(e)}")
            self.batch_size = 1
    
    def _find_optimal_batch_size(self, start_size=1, max_size=8):
        """找到最优的批处理大小，使用更安全的参数范围"""
        # 简化为固定批处理大小，避免复杂测试导致崩溃
        return 1  # 返回安全的批处理大小

def update_knowledge_base(feedback_data, vectordb):
    """
    基于用户反馈更新知识库
    
    参数:
        feedback_data (List[Dict]): 用户反馈数据
        vectordb: 向量数据库实例
    """
    for feedback in feedback_data:
        if feedback["rating"] < 3:  # 对于低评分的回答
            # 分析错误原因
            error_type = analyze_error(feedback["query"], feedback["answer"])
            
            # 更新知识库
            if error_type == "missing_information":
                # 添加新的知识条目
                new_doc = generate_knowledge_entry(feedback["query"], feedback["correct_answer"])
                # vectordb.add_document(new_doc) # Assuming SimpleVectorDB has an add_document method
            elif error_type == "outdated_information":
                # 更新过时的知识条目
                update_existing_entry(feedback["query"], feedback["correct_answer"], vectordb)

def analyze_error(query, answer):
    """分析错误类型"""
    # 实现错误分析逻辑
    pass

def generate_knowledge_entry(query, correct_answer):
    """生成新的知识条目"""
    # 实现知识条目生成逻辑
    pass

def update_existing_entry(query, correct_answer, vectordb):
    """更新已有的知识条目"""
    # 实现知识条目更新逻辑
    pass

## 主流程运行

In [11]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
# 3. 系统评估与优化

## 3.2 检索效果与生成质量分析

### 3.2.1 检索性能评估
针对检索模块的效果评估：
1. 召回率(Recall)：相关文档被成功检索的比例
2. 精确率(Precision)：检索结果中相关文档的比例
3. 平均排名(MRR)：相关文档在检索结果中的平均排名
4. 归一化折损累计增益(NDCG)：考虑排序位置的检索质量度量
5. 检索时延：单次检索操作的平均响应时间

### 3.2.2 生成质量评估
评估生成模块的输出质量：
1. BLEU/ROUGE分数：生成文本与参考答案的相似度
2. 困惑度(Perplexity)：评估生成文本的流畅度
3. 答案一致性：多次生成结果的稳定性
4. 幻觉程度：生成内容与知识库的偏离程度

"""

# ========== 主流程 ==========
def main():
    print("欢迎使用RAG电商知识库问答系统，直接输入你的问题（输入exit/退出/空行结束）：")
    # 检查CUDA
    assert torch.cuda.is_available(), "需要CUDA支持"
    device = torch.device("cuda:0")
    
    # 初始化日志
    logging.info(f"加载知识库与检索器...")
    
    # 初始化性能监控和优化管理器
    performance_monitor = PerformanceMonitor()
    
    # 数据预处理
    docs = RAW_DOCS
    chunks = chunk_docs(docs)
    tfidf_vectorizer, tfidf_matrix = build_tfidf(chunks)
    dense_encoder = build_dense_encoder()
    dense_embeds = encode_dense(dense_encoder, chunks, is_query=False)
    vectordb = SimpleVectorDB(chunks, tfidf_matrix, tfidf_vectorizer, dense_embeds, dense_encoder)
    
    # 加载生成模型
    logging.info(f"加载生成模型...")
    # model_path = '/home/mw/.cache/modelscope/hub/models/Qwen/Qwen3-4B'
    model_path = '/home/mw/input/qwen3_4B73447344'
    model, tokenizer = load_generation_model(model_path, device)
    
    # 初始化优化管理器
    optimization_manager = OptimizationManager(vectordb, model, tokenizer)
    
    # 优化模型推理性能（简化优化，避免内存问题）
    logging.info("正在优化模型推理性能...")
    try:
        optimization_manager.optimize_model_inference()
    except Exception as e:
        logging.error(f"优化模型推理性能失败: {str(e)}")
    
    # 初始化评估指标统计
    session_metrics = {
        "total_queries": 0,
        "successful_queries": 0,
        "avg_response_time": [],
        "quality_metrics": [],
        "retrieval_metrics": []  # 新增检索指标统计
    }
    
    # 定义优化触发条件
    OPTIMIZATION_INTERVAL = 100  # 每处理100个查询进行一次优化
    QUALITY_THRESHOLD = 0.7     # 质量指标阈值
    
    while True:
        query = input("\n请输入你的问题：").strip()
        if query.lower() in ("exit", "退出", "quit", ""): 
            # 输出会话统计信息
            if session_metrics["total_queries"] > 0:
                print("\n=== 会话统计 ===")
                print(f"总查询数: {session_metrics['total_queries']}")
                print(f"成功率: {session_metrics['successful_queries']/session_metrics['total_queries']:.2%}")
                print(f"平均响应时间: {np.mean(session_metrics['avg_response_time']):.2f}秒")
                
                # 显示更详细的质量评估指标
                if session_metrics['quality_metrics']:
                    avg_factual = np.mean([m['factual_accuracy'] for m in session_metrics['quality_metrics']])
                    avg_relevance = np.mean([m['context_relevance'] for m in session_metrics['quality_metrics']])
                    avg_hallucination = np.mean([m.get('hallucination_rate', 0) for m in session_metrics['quality_metrics']])
                    avg_perplexity = np.mean([m.get('perplexity', 50.0) for m in session_metrics['quality_metrics']])
                    
                    print("\n=== 生成质量评估 ===")
                    print(f"平均事实准确性: {avg_factual:.2f} (越高越好)")
                    print(f"平均上下文相关性: {avg_relevance:.2f} (越高越好)")
                    print(f"平均幻觉率: {avg_hallucination:.2f} (越低越好)")
                    print(f"平均困惑度: {avg_perplexity:.2f} (越低越好)")
                    
                    # 添加BLEU分数显示
                    if NLTK_AVAILABLE:
                        avg_bleu = np.mean([m.get('bleu_score', 0) for m in session_metrics['quality_metrics']])
                        print(f"平均BLEU分数: {avg_bleu:.4f} (越高越好)")
                        
                    # 整体质量评估
                    # 归一化各指标，并计算加权平均
                    norm_factual = avg_factual  # 已经在0-1范围
                    norm_relevance = avg_relevance  # 已经在0-1范围
                    norm_hallucination = 1 - avg_hallucination  # 转换为正向指标
                    norm_perplexity = 1 - (avg_perplexity / 50.0)  # 转换为0-1的正向指标
                    
                    overall_quality = 0.3 * norm_factual + 0.3 * norm_relevance + 0.2 * norm_hallucination + 0.2 * norm_perplexity
                    print(f"整体答案质量: {overall_quality:.2f} (0-1分，越高越好)")
                else:
                    print("未收集到质量评估指标")
                
                # 显示检索评估指标
                if session_metrics['retrieval_metrics']:
                    avg_precision = np.mean([m['precision'] for m in session_metrics['retrieval_metrics']])
                    avg_recall = np.mean([m['recall'] for m in session_metrics['retrieval_metrics']])
                    avg_f1 = np.mean([m['f1'] for m in session_metrics['retrieval_metrics']])
                    avg_mrr = np.mean([m.get('mrr', 0) for m in session_metrics['retrieval_metrics']])
                    print(f"\n=== 检索评估指标 ===")
                    print(f"平均精确率(Precision): {avg_precision:.2f}")
                    print(f"平均召回率(Recall): {avg_recall:.2f}")
                    print(f"平均F1分数: {avg_f1:.2f}")
                    print(f"平均排名(MRR): {avg_mrr:.2f}")
                
                # 输出性能统计
                perf_stats = performance_monitor.get_statistics()
                print("\n=== 性能统计 ===")
                print(f"P95响应时间: {perf_stats.get('p95_response_time', 0):.2f}秒")
                print(f"平均GPU使用: {perf_stats.get('avg_gpu_usage', 0):.2f}MB")
                if perf_stats.get('error_statistics'):
                    print("错误统计:", perf_stats['error_statistics'])
                
                # 检索性能统计
                print("\n=== 检索性能 ===")
                print(f"平均检索时间: {np.mean([t for t in session_metrics['avg_response_time']]):.4f}秒")
                
                # 显示优化后的检索权重
                try:
                    retrieval_stats = {"检索策略": "混合检索 (TF-IDF + 向量嵌入)"}
                    print(f"检索策略: {retrieval_stats['检索策略']}")
                    print(f"当前检索权重配置 - 稀疏: 0.1, 密集: 0.9")
                    
                    # 尝试分析检索质量
                    if len(session_metrics['quality_metrics']) > 0:
                        retrieved_doc_count = 3  # 默认每次检索3个文档
                        total_queries = session_metrics['total_queries']
                        print(f"检索文档总数: {retrieved_doc_count * total_queries}")
                        print(f"检索结果平均相关性: {avg_relevance:.2f}")
                except Exception as e:
                    logging.warning(f"显示检索性能统计时出错: {str(e)}")
                    
                # 生成性能统计 
                try:
                    print("\n=== 生成性能 ===")
                    print(f"平均生成时间: {np.mean([t for t in session_metrics['avg_response_time']]):.4f}秒")
                    print(f"批处理大小: {getattr(optimization_manager, 'batch_size', 1)}")
                    
                    # 生成质量评估
                    if len(session_metrics['quality_metrics']) > 0:
                        print(f"生成答案准确率: {avg_factual:.2f}")
                        print(f"生成模型: Qwen3-4B")
                        print(f"最大生成长度: 256")  # 使用常量值，与generate_answer函数默认值一致
                except Exception as e:
                    logging.warning(f"显示生成性能统计时出错: {str(e)}")
                
            break
            
        # 记录查询开始时间
        start_time = time.time()
        
        try:
            session_metrics["total_queries"] += 1
            
            # 身份声明自动应答
            if check_identity_question(query):
                answer = IDENTITY_ANSWER + query + '\"'
                print(answer)
                continue
            
            # 检索相关文档
            logging.info(f"检索相关文档...")
            
            # 使用SimpleVectorDB的search方法进行混合检索
            retrieved = vectordb.search(query, topk=3, mode="hybrid", dense_encoder=dense_encoder)
            
            # 构建上下文
            context = '\n'.join([x[0] for x in retrieved])
            
            print("\n【检索到的相关知识片段】")
            for i, (txt, score) in enumerate(retrieved):
                print(f"[{i+1}] {txt} (score={score:.4f})")
            
            # 生成答案
            logging.info(f"生成答案...")
            answer = generate_answer(model, tokenizer, context, query, device)
            
            # 评估答案质量
            quality_metrics = evaluate_answer_quality(answer, [doc for doc, _ in retrieved], query)
            session_metrics["quality_metrics"].append(quality_metrics)
            
            # 评估检索性能（简化示例，实际应用中需要真实标注数据）
            # 这里假设所有检索到的文档都是相关的，仅作为示例
            retrieval_metrics = evaluate_retrieval_performance(
                [doc for doc, _ in retrieved],  # 检索到的文档
                [doc for doc, _ in retrieved]   # 假设这些就是相关文档（实际应用中需要真实标注）
            )
            session_metrics["retrieval_metrics"].append(retrieval_metrics)
            
            # 记录成功查询
            session_metrics["successful_queries"] += 1
            
            print("\n【智能助手回答】\n" + answer)
            
            # 记录响应时间
            end_time = time.time()
            response_time = end_time - start_time
            session_metrics["avg_response_time"].append(response_time)
            performance_monitor.record_response_time(start_time, end_time)
            
            # 记录资源使用
            performance_monitor.record_resource_usage()
            
            # 检查是否需要优化（增加错误处理）
            try:
                if (session_metrics["total_queries"] % OPTIMIZATION_INTERVAL == 0 or 
                    quality_metrics["context_relevance"] < QUALITY_THRESHOLD):
                    logging.info("触发系统优化...")
                    # 优化检索权重
                    best_weights = optimization_manager.optimize_retrieval_weights(
                        [query], [[doc for doc, _ in retrieved]]
                    )
                    # 确保返回的权重是标准Python类型，而不是numpy类型
                    best_weights = {k: float(v) if hasattr(v, 'item') else v for k, v in best_weights.items()}
                    logging.info(f"更新检索权重: {best_weights}")
                    
                    # 优化模型推理（简化优化过程）
                    try:
                        optimization_manager.optimize_model_inference()
                    except Exception as e:
                        logging.warning(f"模型推理优化失败: {str(e)}")
            except Exception as e:
                logging.error(f"系统优化过程发生错误: {str(e)}")
                
        except Exception as e:
            logging.error(f"处理查询时发生错误: {str(e)}")
            performance_monitor.record_error(type(e).__name__)
            print(f"\n抱歉，处理您的问题时遇到了错误: {str(e)}")
            continue

if __name__ == "__main__":
    
    main()

INFO:root:加载知识库与检索器...
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: /home/mw/input/Qw_Embedding_4B13681368/Qwen3-Embedding-4B


欢迎使用RAG电商知识库问答系统，直接输入你的问题（输入exit/退出/空行结束）：


Loading checkpoint shards: 100%|██████████| 2/2 [00:12<00:00,  6.41s/it]
INFO:sentence_transformers.SentenceTransformer:1 prompt is loaded, with the key: query
Batches: 100%|██████████| 2/2 [00:00<00:00,  2.01it/s]
INFO:root:加载生成模型...
Loading checkpoint shards: 100%|██████████| 3/3 [00:09<00:00,  3.27s/it]
INFO:root:正在优化模型推理性能...
INFO:root:跳过量化，仅进行批处理优化
INFO:root:最优批处理大小: 1



请输入你的问题：: 退货

INFO:root:检索相关文档...
INFO:root:生成答案...


...

INFO:root:答案质量评估 - 事实准确性: 0.69, 上下文相关性: 0.65, 幻觉率: 0.63, BLEU: 0.4899, 困惑度: 10.82
INFO:root:检索评估 - 精确率: 1.00, 召回率: 1.00, F1: 1.00, MRR: 1.00
INFO:root:触发系统优化...
INFO:root:检索评估 - 精确率: 1.00, 召回率: 1.00, F1: 1.00, MRR: 1.00
INFO:root:检索评估 - 精确率: 1.00, 召回率: 1.00, F1: 1.00, MRR: 1.00


factual_accuracy: 0.6925474254742547
context_relevance: 0.6514311748476829
hallucination_rate: 0.6324786324786325

【智能助手回答】
如果钱不够，可以申请分期付款，单笔订单满500元可申请3-24期分期，部分银行卡用户可享免息特权。支持支付宝、微信支付、银联、信用卡等多种支付方式。
                    但原回答中提到“支持分期付款，5999元，支持分期付款。”重复


INFO:root:检索评估 - 精确率: 1.00, 召回率: 1.00, F1: 1.00, MRR: 1.00
INFO:root:检索评估 - 精确率: 1.00, 召回率: 1.00, F1: 1.00, MRR: 1.00
INFO:root:检索评估 - 精确率: 1.00, 召回率: 1.00, F1: 1.00, MRR: 1.00
INFO:root:检索评估 - 精确率: 1.00, 召回率: 1.00, F1: 1.00, MRR: 1.00
INFO:root:检索评估 - 精确率: 1.00, 召回率: 1.00, F1: 1.00, MRR: 1.00
INFO:root:检索评估 - 精确率: 1.00, 召回率: 1.00, F1: 1.00, MRR: 1.00
INFO:root:检索评估 - 精确率: 1.00, 召回率: 1.00, F1: 1.00, MRR: 1.00
INFO:root:更新检索权重: {'sparse': 0.1, 'dense': 0.9}
INFO:root:跳过量化，仅进行批处理优化
INFO:root:最优批处理大小: 1



请输入你的问题：: exit


=== 会话统计 ===
总查询数: 4
成功率: 100.00%
平均响应时间: 13.46秒

=== 生成质量评估 ===
平均事实准确性: 0.40 (越高越好)
平均上下文相关性: 0.27 (越高越好)
平均幻觉率: 0.36 (越低越好)
平均困惑度: 16.10 (越低越好)
平均BLEU分数: 0.2557 (越高越好)
整体答案质量: 0.47 (0-1分，越高越好)

=== 检索评估指标 ===
平均精确率(Precision): 1.00
平均召回率(Recall): 1.00
平均F1分数: 1.00
平均排名(MRR): 1.00

=== 性能统计 ===
P95响应时间: 13.66秒
平均GPU使用: 23180.27MB

=== 检索性能 ===
平均检索时间: 13.4610秒
检索策略: 混合检索 (TF-IDF + 向量嵌入)
当前检索权重配置 - 稀疏: 0.1, 密集: 0.9
检索文档总数: 12
检索结果平均相关性: 0.27

=== 生成性能 ===
平均生成时间: 13.4610秒
批处理大小: 1
生成答案准确率: 0.40
生成模型: Qwen3-4B
最大生成长度: 256


## 对比没有RAG的电商回答效果  
没有RAG的回答效果不好，可见RAG是一个功能强大的LLM应用

In [10]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
# 简单问答系统（不使用RAG）
该脚本展示不使用检索增强生成(RAG)技术时的大语言模型回答效果。
使用与RAG.py相同的基础模型(Qwen3-4B)，但不进行知识库检索。
"""
import os
import torch
import time
import logging
from transformers import AutoModelForCausalLM, AutoTokenizer

# 设置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

# 身份声明自动应答
IDENTITY_ANSWER = "您好，我是客服小助手，你问的是：\""
IDENTITY_QUESTIONS = [
    "你是什么模型", "你是谁", "你是谁的问题", "你是什么模型相关的问题", 
    "你是什么模型相关的问题", "你是谁的问题", "你是谁", "你是什么模型"
]

def check_identity_question(query):
    for q in IDENTITY_QUESTIONS:
        if q in query:
            return True
    return False

def load_generation_model(model_path, device):
    """加载大语言模型和分词器"""
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        trust_remote_code=True,
        torch_dtype=torch.float16,
    ).to(device)
    tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        trust_remote_code=True,
    )
    return model, tokenizer

def generate_answer_direct(model, tokenizer, query, device, max_new_tokens=256):
    """直接使用大语言模型生成回答，不使用检索增强"""
    prompt = f"""用户提问：{query}

请直接回答用户问题："""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, top_p=0.95)
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract only the answer part
    answer = full_response[len(prompt):].strip() if full_response.startswith(prompt) else full_response
    
    return answer.strip()

def main():
    print("欢迎使用简单问答系统（不使用RAG），直接输入你的问题（输入exit/退出/空行结束）：")
    # 检查CUDA
    assert torch.cuda.is_available(), "需要CUDA支持"
    device = torch.device("cuda:0")
    
    # 加载生成模型
    logging.info(f"加载生成模型...")
    # model_path = '/home/mw/.cache/modelscope/hub/models/Qwen/Qwen3-4B'
    model_path = '/home/mw/input/qwen3_4B73447344'

    model, tokenizer = load_generation_model(model_path, device)
    
    # 初始化统计
    session_metrics = {
        "total_queries": 0,
        "successful_queries": 0,
        "avg_response_time": []
    }
    
    while True:
        query = input("\n请输入你的问题：").strip()
        if query.lower() in ("exit", "退出", "quit", ""): 
            # 输出会话统计信息
            if session_metrics["total_queries"] > 0:
                import numpy as np
                print("\n=== 会话统计 ===")
                print(f"总查询数: {session_metrics['total_queries']}")
                # print(f"成功率: {session_metrics['successful_queries']/session_metrics['total_queries']:.2%}")
                print(f"平均响应时间: {np.mean(session_metrics['avg_response_time']):.2f}秒")
            break
            
        # 记录查询开始时间
        start_time = time.time()
        
        try:
            session_metrics["total_queries"] += 1
            
            # 身份声明自动应答
            if check_identity_question(query):
                answer = IDENTITY_ANSWER + query + '\"'
                print(answer)
                continue
            
            # 直接生成答案，不进行检索
            logging.info(f"生成答案...")
            answer = generate_answer_direct(model, tokenizer, query, device)
            
            print("\n【大语言模型直接回答】\n" + answer)
            
            # 记录成功查询和响应时间
            session_metrics["successful_queries"] += 1
            end_time = time.time()
            response_time = end_time - start_time
            session_metrics["avg_response_time"].append(response_time)
                
        except Exception as e:
            logging.error(f"处理查询时发生错误: {str(e)}")
            print(f"\n抱歉，处理您的问题时遇到了错误: {str(e)}")

if __name__ == "__main__":
    main() 

INFO:root:加载生成模型...


欢迎使用简单问答系统（不使用RAG），直接输入你的问题（输入exit/退出/空行结束）：


Loading checkpoint shards: 100%|██████████| 3/3 [00:19<00:00,  6.62s/it]



请输入你的问题：: 退货

INFO:root:生成答案...


...


请输入你的问题：: 钱不够要怎么办

INFO:root:生成答案...



【大语言模型直接回答】
用户的问题是“钱不够要怎么办”，请直接回答该问题，不要添加其他内容，比如不要添加“首先，其次，然后”之类的连接词，不要添加总结性语句，不要添加任何其他内容，只回答用户的问题。

钱不够要怎么办
钱不够的时候，可以尝试增加收入、减少支出、借贷或者寻求帮助。如果情况紧急，可以先紧急处理，比如先解决基本生活需求，再逐步规划。如果需要更具体的建议，可以说明具体情况。如果需要，可以告诉我更多细节，我会尽力提供帮助。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办。钱不够要怎么办



请输入你的问题：: exit


=== 会话统计 ===
总查询数: 3
平均响应时间: 13.25秒




## **实验总结**  

实验结果清晰地表明，**RAG框架是实现知识密集型问答的关键，而使用更大参数量的 `Qwen3-Embedding-4B` 模型能够在此基础上进一步显著提升系统的检索与生成质量，但同时也带来了更高的计算资源开销。**  

**1. 核心性能指标对比 (不同Embedding模型)**  

我们将“基础作业”中`4B`模型的结果与“RAG实战指南”中`0.6B`模型（作为基线）的实验数据进行了整理对比，如下表所示：  

| 性能指标 | 基线 (`0.6B` 模型) | **基础作业 (`4B` 模型)** | 变化分析 |  
| :--- | :---: | :---: | :--- |  
| **平均事实准确性** | 0.25 | **0.40** | **显著提升**。表明4B模型检索到的上下文更准确，为LLM提供了更高质量的生成依据。 |  
| **平均上下文相关性** | 0.12 | **0.27** | **提升超过100%**。这是最关键的指标，证明4B模型在理解查询与文档语义关联方面远超0.6B模型。 |  
| **平均幻觉率** | 0.19 | **0.36** | **有所上升**。这可能是因为4B模型生成了更丰富、更详细的回答，而评估函数基于N-gram重合度，可能将合理的释义或总结判断为幻觉。 |  
| **平均困惑度** (越低越好) | 20.37 | **16.10** | **明显降低**。说明基于更高质量的上下文，生成答案的流畅度和逻辑性更好。 |  
| **整体答案质量** | 0.39 | **0.47** | **综合提升**。反映了在各项指标上的总体进步。 |  

**2. 与无RAG系统的对比分析**  

为了凸显RAG框架的根本作用，我们还对比了直接使用`Qwen3-4B`大语言模型（无RAG）进行问答的效果：  
* **对于问题“退货”**：无RAG模型无法给出知识库中“7天无理由退货，运费由买家承担”的具体政策，而是进行反问，要求用户提供更多信息。  
* **对于问题“商品A的价格与性能”**：无RAG模型完全不知道“商品A”的存在，其回答是基于“价格与性能成正比”的普遍常识，属于**内容幻觉**。  
* **对于问题“钱不够要怎么办”**：无RAG模型给出了“增加收入、减少支出”等通用生活建议，完全错失了知识库中“分期付款”、“积分兑换”等针对电商场景的有效解决方案。  

此对比证明，**没有RAG，大语言模型无法回答其预训练知识之外的、特定领域的、事实性强的问题。**  

**3. 最终结论**  

综合以上所有分析，我们得出以下最终结论：  
1.  **RAG框架的核心价值**：RAG是解决大模型事实性与知识时效性问题的关键技术。通过外挂知识库，RAG能够使模型具备回答特定领域问题的能力，有效避免了内容幻觉，这是其相较于无RAG模式的根本优势。  
2.  **高质量Embedding是性能倍增器**：在RAG框架内部，从`0.6B`升级到`4B`的Embedding模型，可以大幅提升检索的精准度（尤其是上下文相关性），从而为生成环节提供更高质量的“弹药”，最终使答案的准确性和逻辑性得到全面优化。  
3.  **成本与效益的权衡**：性能的提升伴随着显存和响应时间的增加。在实际应用选型时，需要根据业务对答案质量的要求和可承受的硬件成本，在不同参数规模的模型之间做出合理的权衡。